# 09 Ablation Experiments

This notebook organizes the ablation analysis for the project. It runs executable structured-feature ablations immediately and records the planned text-only and multimodal fusion ablations so the experiment matrix stays publication-ready.

## Ablation scope

- Vitals only
- Vitals plus labs
- Full structured EHR
- Planned text-only comparison
- Planned multimodal fusion comparison

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/multimodal-early-sepsis') if IN_COLAB else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
if IN_COLAB:
    %pip -q install pyyaml pandas numpy scikit-learn xgboost

In [ ]:
import pandas as pd

from src.evaluation.ablation import (
    build_fusion_strategy_table,
    build_planned_ablation_matrix,
    run_structured_ablation_suite,
)
from src.utils.config import load_config
from src.utils.io_utils import save_dataframe_bundle
from src.utils.logging_utils import write_run_manifest
from src.utils.paths import ensure_directories, resolve_project_paths

In [ ]:
base_config_path = PROJECT_ROOT / 'configs' / 'base.yaml'
override_config_path = PROJECT_ROOT / 'configs' / 'colab.yaml' if IN_COLAB else None
config = load_config(base_config_path, override_config_path)
paths = resolve_project_paths(config)
ensure_directories(paths)
config['ablation']

## Load horizon datasets and multimodal experiment plan

In [ ]:
horizon_tables = {}
for horizon in config['prediction']['horizons_hours']:
    dataset_name = f'horizon_{int(horizon)}h'
    path = paths['processed_data_dir'] / '04_feature_engineering' / f'{dataset_name}.csv'
    assert path.exists(), f'Missing structured horizon dataset: {dataset_name}. Run 04_feature_engineering first.'
    horizon_tables[dataset_name] = pd.read_csv(path, parse_dates=['hour', 'prediction_time', 'INTIME', 'OUTTIME'])

multimodal_plan_path = paths['processed_data_dir'] / '07_multimodal_models' / 'multimodal_experiment_plan.csv'
multimodal_plan_df = pd.read_csv(multimodal_plan_path) if multimodal_plan_path.exists() else pd.DataFrame()
{key: value.shape for key, value in horizon_tables.items()}, multimodal_plan_df.shape

## Run executable structured ablations

In [ ]:
artifacts = run_structured_ablation_suite(horizon_tables, config)
ablation_results_df = artifacts['ablation_results']
ablation_results_df

## Build planned ablation matrix and fusion strategy table

In [ ]:
planned_matrix_df = build_planned_ablation_matrix(config)
fusion_strategy_df = build_fusion_strategy_table(multimodal_plan_df)
planned_matrix_df, fusion_strategy_df

## Inspect strongest ablation results

In [ ]:
ablation_results_df.sort_values('auprc', ascending=False).head(12) if not ablation_results_df.empty else ablation_results_df

## Save ablation artifacts

In [ ]:
output_dir = paths['processed_data_dir'] / '09_ablation_experiments'
artifact_bundle = dict(artifacts)
artifact_bundle['planned_ablation_matrix'] = planned_matrix_df
artifact_bundle['fusion_strategy_table'] = fusion_strategy_df
saved_paths = save_dataframe_bundle(artifact_bundle, output_dir)
saved_paths

In [ ]:
manifest_path = paths['manifests_dir'] / '09_ablation_experiments_manifest.json'
write_run_manifest(
    path=manifest_path,
    stage='09_ablation_experiments',
    config=config,
    extra={
        'saved_artifacts': saved_paths,
        'multimodal_plan_available': bool(len(multimodal_plan_df)),
        'ablation_result_rows': int(len(ablation_results_df)),
    },
)
manifest_path

## Review checklist

Before explainability analysis, review:

- How much performance comes from vitals alone?
- Do labs add meaningful signal beyond vitals?
- Which horizons benefit most from richer structured features?
- Which fusion strategies should be prioritized once full multimodal training outputs are available?
- Is the planned ablation matrix complete enough for the paper?

## Next notebook

`10_explainability.ipynb` will prepare SHAP-based structured explanations, attention-based note interpretation, and clinically meaningful narratives for the final report.